In [ ]:
from sage.all import *
from sage.plot.plot3d.shapes import LineSegment

R = 200
r = 2   # geometric radius in coordinate units
p = 50
h = 80  # vertical scale

# 7 vertices → 6-line chain
z_offsets = [0, 1, -0.5, 1.2, -1.5, 0.7, -1.8]  # chosen so points are not coplanar
vertices = [
    (R*cos(2*pi*k/7), R*sin(2*pi*k/7), h*z_offsets[k])
    for k in range(7)
]

# Build extended "full" lines through consecutive vertex pairs
extend_factor = 2  # how far past each vertex to extend
segments = []
for k in range(6):           # 6 lines: l1,...,l6
    v1 = vector(vertices[k])
    v2 = vector(vertices[k+1])
    d = v2 - v1
    p_ext = tuple(v1 - extend_factor * d)
    q_ext = tuple(v2 + extend_factor * d)
    segments.append((p_ext, q_ext))

# Lines
G_lines = sum(LineSegment(p, q, radius=r, color='black')
              for p, q in segments)

# Spheres at intersection points (inner vertices v1,...,v5)
G_spheres = sum(sphere(vertices[k], r, color='red') for k in range(1, 6))

# Planes H1 (l1,l2), H2 (l3,l4), H3 (l5,l6)
u, v = var('u v')
plane_extent = 1200

def plane_patch(segA, segB, color):
    p1, q1 = map(vector, segA)
    p2, q2 = map(vector, segB)
    d1 = (q1 - p1).normalized()
    d2 = (q2 - p2).normalized()
    base = (p1 + p2) / 2   # point in the plane

    fx = lambda u, v: base[0] + u*d1[0] + v*d2[0]
    fy = lambda u, v: base[1] + u*d1[1] + v*d2[1]
    fz = lambda u, v: base[2] + u*d1[2] + v*d2[2]

    return parametric_plot3d(
        (fx, fy, fz),
        (u, -plane_extent, plane_extent),
        (v, -plane_extent, plane_extent),
        color=color, opacity=0.3, mesh=False
    )

l1, l2, l3, l4, l5, l6 = segments
H1 = plane_patch(l1, l2, color='blue')
H2 = plane_patch(l3, l4, color='green')
H3 = plane_patch(l5, l6, color='purple')

show(G_lines + G_spheres + H1 + H2 + H3, frame=False)